# Quickstart — Browserless × LlamaIndex (Python)

Connect to the Browserless MCP server, load its browsing tools, call one directly, and run a `FunctionAgent` that scrapes the web.

Get a token at [account.browserless.io](https://account.browserless.io).

In [ ]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

## Connect and load tools

- Pass `timeout=120`: the default 30 s is too short for the first call's MCP handshake plus browser warm-up.
- Browserless tool schemas include an optional `_prompt` field, and Pydantic rejects field names that start with an underscore. `BrowserlessToolSpec` skips those fields so `McpToolSpec` can build every tool.
- The server also exposes account tools and widget-only tools for MCP Apps clients; `allowed_tools` keeps just the browsing tools.

In [ ]:
import os

from llama_index.tools.mcp import BasicMCPClient, McpToolSpec


class BrowserlessToolSpec(McpToolSpec):
    # Pydantic rejects field names with a leading underscore, so skip
    # optional fields such as `_prompt` when building each tool's schema.
    def create_model_from_json_schema(self, schema, model_name="DynamicModel"):
        properties = {
            name: prop
            for name, prop in schema.get("properties", {}).items()
            if not name.startswith("_")
        }
        return super().create_model_from_json_schema(
            {**schema, "properties": properties}, model_name
        )


client = BasicMCPClient(
    os.environ.get("BROWSERLESS_MCP_URL", "https://mcp.browserless.io/mcp"),
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)


tools = await BrowserlessToolSpec(
    client=client,
    allowed_tools=[
        "browserless_smartscraper",
        "browserless_search",
        "browserless_map",
        "browserless_crawl",
        "browserless_export",
        "browserless_performance",
        "browserless_function",
        "browserless_agent",
        "browserless_skill",
    ],
).to_tool_list_async()
print(f"Loaded {len(tools)} tools:")
for t in tools:
    print(f"  - {t.metadata.name}")

## Direct tool invocation

Each MCP tool wraps as a LlamaIndex `FunctionTool`. Call `browserless_smartscraper` directly to fetch a deterministic URL as markdown.

In [ ]:
def text(output):
    return "\n".join(c.text for c in output.raw_output.content if c.type == "text")


smartscraper = next(t for t in tools if t.metadata.name == "browserless_smartscraper")
result = await smartscraper.acall(url="https://example.com", formats=["markdown"])
print(text(result)[:500])

## FunctionAgent

Hand the tools to a LlamaIndex `FunctionAgent` powered by Claude. The model picks which tool to call at each step.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

agent = FunctionAgent(
    name="BrowserlessAgent",
    description="Browses and scrapes the web via the Browserless MCP server.",
    llm=Anthropic(model="claude-sonnet-5-5"),
    tools=tools,
    system_prompt="You are a helpful research assistant that uses the Browserless tools to scrape and summarize the web.",
)

response = await agent.run(
    "Scrape https://news.ycombinator.com and list the top 5 headlines as a markdown bullet list."
)
print(response)